# 05 — EDA-traceable purchase-time features

This notebook reads all three fixed splits but fits the preprocessor on training rows only. The whitelist is tied directly to the retained groups in `findings.md`. Availability at purchase—not predictive strength alone—is the admission rule.

In [1]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert ROOT.name == "02-olist-late-delivery-ml"
SEED = 42
pd.set_option("display.max_columns", 100)
import holidays
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

parts = {
    name: pd.read_parquet(ROOT / f"artifacts/03_splits/{name}.parquet")
    for name in ["train", "validation", "test"]
}
findings = (ROOT / "artifacts/04_eda/findings.md").read_text()
required_evidence = [
    "promised_window_days", "freight_price_ratio", "purchase-holiday",
    "Haversine", "same-state",
]
assert all(term in findings for term in required_evidence)

base_numeric = [
    "item_count", "unique_product_count", "unique_seller_count",
    "total_price", "total_freight", "mean_product_weight_g",
    "mean_product_volume_cm3", "mean_product_photos_qty", "payment_count",
    "payment_value", "max_payment_installments", "same_state_share",
    "mean_customer_seller_distance_km", "max_customer_seller_distance_km",
]
base_categorical = [
    "customer_state", "dominant_product_category",
    "dominant_seller_state", "dominant_payment_type",
]
derived_numeric = [
    "purchase_month", "purchase_weekday", "purchase_hour",
    "promised_window_days", "freight_price_ratio", "purchase_is_holiday",
]
FEATURE_WHITELIST = base_numeric + base_categorical + derived_numeric
FORBIDDEN = {
    "order_delivered_customer_date", "order_delivered_carrier_date",
    "order_approved_at", "order_status", "delivery_delta_hours", "late",
    "order_id", "customer_id", "customer_unique_id", "shipping_limit_date",
    "geo_lat", "geo_lng", "customer_zip_code_prefix", "dominant_seller_zip_prefix",
}
assert not set(FEATURE_WHITELIST) & FORBIDDEN
assert not any("review" in column for column in FEATURE_WHITELIST)
FEATURE_WHITELIST

['item_count',
 'unique_product_count',
 'unique_seller_count',
 'total_price',
 'total_freight',
 'mean_product_weight_g',
 'mean_product_volume_cm3',
 'mean_product_photos_qty',
 'payment_count',
 'payment_value',
 'max_payment_installments',
 'same_state_share',
 'mean_customer_seller_distance_km',
 'max_customer_seller_distance_km',
 'customer_state',
 'dominant_product_category',
 'dominant_seller_state',
 'dominant_payment_type',
 'purchase_month',
 'purchase_weekday',
 'purchase_hour',
 'promised_window_days',
 'freight_price_ratio',
 'purchase_is_holiday']

## Deterministic feature construction

Calendar fields use purchase timestamp; holidays use the pinned Brazil national calendar. Promised window uses the customer-facing estimate. Freight ratio uses purchase-time order values. Distances were already computed only from valid endpoints in Notebook 01.

In [2]:
years = sorted(parts["train"].order_purchase_timestamp.dt.year.unique())
brazil_holidays = holidays.Brazil(years=years)

def make_features(frame):
    features = frame.copy()
    purchase = features.order_purchase_timestamp
    features["purchase_month"] = purchase.dt.month
    features["purchase_weekday"] = purchase.dt.weekday
    features["purchase_hour"] = purchase.dt.hour
    features["purchase_is_holiday"] = purchase.dt.date.map(
        lambda date: int(date in brazil_holidays)
    )
    features["promised_window_days"] = (
        features.order_estimated_delivery_date - purchase
    ).dt.total_seconds() / 86_400
    features["freight_price_ratio"] = (
        features.total_freight / features.total_price.replace(0, np.nan)
    )
    return features[FEATURE_WHITELIST]

raw_features = {name: make_features(frame) for name, frame in parts.items()}
labels = {name: frame["late"].astype("int8") for name, frame in parts.items()}

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore", min_frequency=20, sparse_output=False,
    )),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, base_numeric + derived_numeric),
    ("categorical", categorical_pipeline, base_categorical),
], verbose_feature_names_out=False)

X_train = preprocessor.fit_transform(raw_features["train"])
X_validation = preprocessor.transform(raw_features["validation"])
X_test = preprocessor.transform(raw_features["test"])
feature_names = preprocessor.get_feature_names_out().tolist()

assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1]
assert len(feature_names) == X_train.shape[1]
assert np.isfinite(X_train).all()
assert np.isfinite(X_validation).all()
assert np.isfinite(X_test).all()
X_train.shape, X_validation.shape, X_test.shape

((67533, 135), (14471, 135), (14472, 135))

## Persist the preprocessing contract

Dense Parquet is suitable at this scale. Metadata records the exact whitelist and fit scope so leakage can be audited without relying on notebook prose.

In [3]:
out = ROOT / "artifacts/05_features"
out.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, out / "preprocessor.joblib")

transformed = {"train": X_train, "validation": X_validation, "test": X_test}
for name, matrix in transformed.items():
    pd.DataFrame(matrix, columns=feature_names).to_parquet(
        out / f"X_{name}.parquet", index=False
    )
    labels[name].rename("late").to_frame().to_parquet(
        out / f"y_{name}.parquet", index=False
    )

(out / "feature_list.json").write_text(json.dumps(feature_names, indent=2))
(out / "feature_selection_metadata.json").write_text(json.dumps({
    "prediction_time": "order purchase timestamp",
    "whitelist": FEATURE_WHITELIST,
    "eda_trace": "artifacts/04_eda/findings.md retained feature groups",
    "forbidden": sorted(FORBIDDEN),
    "fit_scope": "train only",
    "train_rows_fit": len(parts["train"]),
    "validation_action": "transform only",
    "test_action": "transform only",
}, indent=2))
{"transformed_features": len(feature_names), "fit_rows": len(parts["train"])}

{'transformed_features': 135, 'fit_rows': 67533}